In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import pickle
import tensorflow_datasets as tfds
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
BASE_PATH = Path('/content/drive/MyDrive/AnanthiX_AI')
DATA_PATH = BASE_PATH / 'data'
RESULTS_PATH = BASE_PATH / 'results'
MODELS_PATH = BASE_PATH / 'models'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Chargement metadata')
with open(DATA_PATH / 'plantvillage_metadata.json', 'r') as f:
    metadata = json.load(f)

class_names = metadata['class_names']
num_classes = len(class_names)

print(f'Classes: {num_classes}')

print('Chargement dataset')
ds, info = tfds.load('plant_village', with_info=True, as_supervised=True)
train_ds = ds['train']

all_images = []
all_labels = []

for i, (image, label) in enumerate(train_ds):
    all_images.append(image.numpy().astype(np.uint8))
    all_labels.append(label.numpy())
    if (i + 1) % 10000 == 0:
        print(f'{i + 1}/54303')

all_images = np.array(all_images)
all_labels = np.array(all_labels)

print('Dataset chargé')

print('Création splits')

train_indices = []
val_indices = []
test_indices = []

for class_idx in range(num_classes):
    mask = all_labels == class_idx
    indices = np.where(mask)[0]
    n = len(indices)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)

    train_indices.extend(indices[:n_train])
    val_indices.extend(indices[n_train:n_train + n_val])
    test_indices.extend(indices[n_train + n_val:])

print('Splits créés')

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

class FastDataset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        image = self.images[actual_idx]
        label = self.labels[actual_idx]

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, label

test_dataset = FastDataset(all_images, all_labels, test_indices, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)

print(f'Test loader créé: {len(test_loader)} batches')

print('Chargement modèle')

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(2048, num_classes)
model.load_state_dict(torch.load(MODELS_PATH / 'resnet50_baseline_weighted.pth', map_location=device))
model = model.to(device)
model.eval()

print('Modèle chargé')

print('Évaluation complète')

all_preds = []
all_labels_true = []
all_confidence = []
all_images_wrong = []
all_wrong_preds = []
all_wrong_truths = []
all_wrong_conf = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels_true.extend(labels.numpy())
        all_confidence.extend(confidence.cpu().numpy())

        wrong_mask = predicted.cpu().numpy() != labels.numpy()
        if wrong_mask.any():
            for i, is_wrong in enumerate(wrong_mask):
                if is_wrong:
                    all_images_wrong.append(images[i].cpu())
                    all_wrong_preds.append(predicted[i].item())
                    all_wrong_truths.append(labels[i].item())
                    all_wrong_conf.append(confidence[i].item())

all_preds = np.array(all_preds)
all_labels_true = np.array(all_labels_true)
all_confidence = np.array(all_confidence)

accuracy = 100 * np.mean(all_preds == all_labels_true)
print(f'Accuracy: {accuracy:.2f}%')

print('Création confusion matrix')

cm = confusion_matrix(all_labels_true, all_preds)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True, ax=ax, xticklabels=False, yticklabels=False)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Vérité', fontsize=12)
ax.set_title('Confusion Matrix - PlantVillage (38×38)', fontsize=14, fontweight='bold')

cm_path = RESULTS_PATH / 'confusion_matrix_full.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()

print(f'Confusion matrix sauvegardée: {cm_path}')

print('Calcul per-class metrics')

per_class_metrics = {}

for class_idx in range(num_classes):
    tp = cm[class_idx, class_idx]
    fp = cm[:, class_idx].sum() - tp
    fn = cm[class_idx, :].sum() - tp
    tn = cm.sum() - tp - fp - fn

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    per_class_metrics[class_names[class_idx]] = {
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'support': int(cm[class_idx].sum())
    }

with open(RESULTS_PATH / 'per_class_metrics.json', 'w') as f:
    json.dump(per_class_metrics, f, indent=2)

print('Per-class metrics sauvegardées')

worst_classes = sorted(per_class_metrics.items(), key=lambda x: x[1]['f1'])[:5]
print()
print('Top 5 worst F1 classes:')
for class_name, metrics in worst_classes:
    print(f'  {class_name}: {metrics["f1"]:.4f}')

print()
print('Identification top erreurs')

if len(all_images_wrong) > 0:
    sorted_indices = np.argsort(all_wrong_conf)[::-1][:20]

    fig, axes = plt.subplots(4, 5, figsize=(16, 12))
    fig.suptitle('Top 20 Erreurs les Plus Confiantes', fontsize=14, fontweight='bold')
    axes = axes.flatten()

    for i, idx in enumerate(sorted_indices):
        if i >= 20:
            break

        ax = axes[i]
        img = all_images_wrong[idx].numpy().transpose(1, 2, 0)
        img = (img - img.min()) / (img.max() - img.min())

        ax.imshow(img)
        pred_name = class_names[all_wrong_preds[idx]]
        true_name = class_names[all_wrong_truths[idx]]
        conf = all_wrong_conf[idx]

        ax.set_title(f'Pred: {pred_name[:15]}\nTrue: {true_name[:15]}\nConf: {conf:.2f}', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    errors_path = RESULTS_PATH / 'top_errors.png'
    plt.savefig(errors_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f'Top erreurs sauvegardées: {errors_path}')
else:
    print('Aucune erreur (100% accuracy)')

print()
print('Sauvegarde finale')

with open(RESULTS_PATH / 'confusion_matrix.pkl', 'wb') as f:
    pickle.dump(cm, f)

evaluation_results = {
    'test_accuracy': float(accuracy),
    'total_samples': int(len(test_indices)),
    'errors': int(len(all_images_wrong)),
    'per_class_metrics': per_class_metrics
}

with open(RESULTS_PATH / 'evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print(f'Confusion matrix: {RESULTS_PATH / "confusion_matrix.pkl"}')
print(f'Per-class metrics: {RESULTS_PATH / "per_class_metrics.json"}')
print(f'Evaluation results: {RESULTS_PATH / "evaluation_results.json"}')

print('ÉTAPE 3 COMPLETED')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Chargement metadata
Classes: 38
Chargement dataset
10000/54303
20000/54303
30000/54303
40000/54303
50000/54303
Dataset chargé
Création splits
Splits créés
Test loader créé: 171 batches
Chargement modèle
Modèle chargé
Évaluation complète
Accuracy: 99.61%
Création confusion matrix
Confusion matrix sauvegardée: /content/drive/MyDrive/AnanthiX_AI/results/confusion_matrix_full.png
Calcul per-class metrics
Per-class metrics sauvegardées

Top 5 worst F1 classes:
  Corn___Cercospora_leaf_spot Gray_leaf_spot: 0.9375
  Corn___Northern_Leaf_Blight: 0.9630
  Potato___Late_blight: 0.9866
  Tomato___Septoria_leaf_spot: 0.9887
  Tomato___Early_blight: 0.9900

Identification top erreurs
Top erreurs sauvegardées: /content/drive/MyDrive/AnanthiX_AI/results/top_errors.png

Sauvegarde finale
Confusion matrix: /content/drive/MyDrive/AnanthiX_AI/results/confusion_matr